<a href="https://colab.research.google.com/github/quentin-garcia-3b1724235/canadian-banks-portfolio-analysis/blob/main/notebooks/canadian_banks_analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#Canadian Big Six Banks: Portfolio Performance and Risk Analysis

This notebook analyzes the performance and risk of a portfolio made of Canada’s six largest banks, using daily market data from Yahoo Finance.

##Section 0 : Setup

###0.1 Install and test yfinance

In [1]:
!pip install --upgrade --quiet yfinance

In [2]:
import yfinance as yf
import pandas as pd

print("yfinance version:", yf.__version__)

test = yf.download("RY.TO", period="5d", auto_adjust=False, progress=False)
test

yfinance version: 1.8.0


Price,Adj Close,Close,High,Low,Open,Volume
Ticker,RY.TO,RY.TO,RY.TO,RY.TO,RY.TO,RY.TO
Date,,,,,,
2026-11-02,286.209991,286.209991,287.529999,284.000000,284.459991,2792000
2026-11-03,285.420013,285.420013,287.000000,283.529999,285.350006,2093600
2026-11-04,292.570007,292.570007,293.670013,287.500000,288.829987,2495100
2026-11-05,285.859985,285.859985,295.489990,285.510010,294.890015,3162000
2026-11-06,284.160004,284.160004,287.299988,283.250000,285.600006,2480300


###0.2 Connect Google Drive

In [4]:
from google.colab import drive
drive.mount("/content/drive")

import os
PROJECT_DIR = "/content/drive/MyDrive/canadian-banks-portfolio"
os.makedirs(f"{PROJECT_DIR}/data", exist_ok=True)
print(os.listdir(PROJECT_DIR))

Mounted at /content/drive
['data']


##Section 1 : Data

Download 20 years of daily prices for the six banks and the benchmark, then save them as CSV files.


###1.1 Parameters

In [5]:
TICKERS = {
    "RY.TO": "RBC",
    "TD.TO": "TD",
    "BNS.TO": "Scotiabank",
    "BMO.TO": "BMO",
    "CM.TO": "CIBC",
    "NA.TO": "National Bank",
}
BENCHMARK = {"XIC.TO": "TSX Composite ETF"}

START_DATE = "2006-01-01"
END_DATE = "2026-01-01"

###1.2 Download daily prices

In [6]:
all_tickers = list(TICKERS) + list(BENCHMARK)

raw = yf.download(
    all_tickers,
    start=START_DATE,
    end=END_DATE,
    auto_adjust=False,
    actions=True,
    progress=False,
)

print(raw.shape)
raw.columns.get_level_values(0).unique()

(5020, 57)


Index(['Adj Close', 'Capital Gains', 'Close', 'Dividends', 'High', 'Low',
       'Open', 'Stock Splits', 'Volume'],
      dtype='object', name='Price')

###1.3 Extract adjusted prices, closing prices and dividends

In [7]:
adj_close = raw["Adj Close"]
close = raw["Close"]
dividends = raw["Dividends"]

adj_close.tail()

Ticker,BMO.TO,BNS.TO,CM.TO,NA.TO,RY.TO,TD.TO,XIC.TO
Date,,,,,,,
2025-12-23,173.018265,97.054573,122.814125,170.928589,228.010574,125.145035,50.292862
2025-12-24,172.989197,97.313927,122.862648,171.532501,227.815720,125.378006,50.234104
2025-12-29,174.356415,97.285103,122.921371,171.885681,229.111420,125.795410,50.067642
2025-12-30,173.328552,97.333138,122.637558,170.374969,228.936035,126.008972,50.029243
2025-12-31,172.843735,97.237091,121.776329,169.325333,227.952118,125.572151,49.773232


###1.4 Check RBC’s recent dividends

In [8]:
ry_dividends = yf.Ticker("RY.TO").dividends
ry_dividends.tail(6)

,Dividends
Date,
2025-07-24 09:30:00-04:00,1.54
2025-10-27 09:30:00-04:00,1.54
2026-01-26 09:30:00-05:00,1.64
2026-04-23 09:30:00-04:00,1.64
2026-07-27 09:30:00-04:00,1.76
2026-10-26 09:30:00-04:00,1.76


###1.5 Effect of dividends on RBC’s growth

In [9]:
ry_close = close["RY.TO"].dropna()
ry_adj = adj_close["RY.TO"].dropna()

price_growth = ry_close.iloc[-1] / ry_close.iloc[0] - 1
total_growth = ry_adj.iloc[-1] / ry_adj.iloc[0] - 1

print(f"RBC price growth (Close):     {price_growth:.0%}")
print(f"RBC total growth (Adj Close): {total_growth:.0%}")

RBC price growth (Close):     410%
RBC total growth (Adj Close): 1006%


Over 2006–2025, RBC’s share price grew by 410%, but its total return with dividends reinvested reached 1,006%. More than half of the total gain therefore came from dividends. This is why all return calculations in this project use adjusted prices: closing prices alone would strongly understate the performance of dividend-paying stocks such as banks. Assumption: dividends are reinvested with no taxes or fees.

###1.6 Save the data to Google Drive

In [11]:
adj_close.to_csv(f"{PROJECT_DIR}/data/adj_close_prices.csv")
close.to_csv(f"{PROJECT_DIR}/data/close_prices.csv")
dividends.to_csv(f"{PROJECT_DIR}/data/dividends.csv")

print(os.listdir(f"{PROJECT_DIR}/data"))

['adj_close_prices.csv', 'close_prices.csv', 'dividends.csv']
